# TradeSense · Component 03 · Protocol B2 · step 4: Transformer v2 (sequence pattern classifier, GPU)

All under the frozen Protocol B2 (stored in `layer3_v3/`) and the amendments log (A1 and **A2**, below). What this notebook does:
1. checks the protocol hash and that notebooks 28 and 29 finished; **TEST-B / TEST-3 are never loaded**;
2. unit tests;
3. selection on VAL-3 only: window W ∈ {16, 32, 64} × (d_model, layers) ∈ {(32,1), (64,1), (64,2)} = 9 candidates, **for each normalisation N0 and N1** (seed 11). Fit rows: TRAIN-3; early stopping and selection: VAL-3. The two new families are **never** used for fitting, early stopping or selection;
4. H-B3 (window): if the selected W > 16, leave-one-family-out (LOFO) models are trained for the selected candidate and for the best W = 16 candidate; the longer window is kept only if it improves LOFO macro recall (6 old families) by ≥ 0.05, otherwise W = 16 stays;
5. final models (3 seeds) for the final candidate of each normalisation; scores of every series are saved;
6. VAL-3 evaluation next to the comparators (XGBoost v2, AE-ctx fmax, robust-z), the union of the three, a chance baseline, LOFO and per-family recall under T1c / T1 / T0;
7. share block.

**Amendment A2 (written before any Transformer v2 result exists, logged in the project doc `protocol_b2_amendments.md`):**
- the selection metric for the Transformer is VAL-3 recall under **T1c** (amendment A1: T1 is depressed by the dense synthetic VAL-3 series, which made the XGBoost selection noise-dominated). T1 and T0 selections are replayed and reported; any difference is reported openly, nothing is switched silently;
- for speed, at every epoch the clean negatives are subsampled at random (fraction `NEG_FRAC` = 0.5, a new draw each epoch); all positives are used. Everything else is as v1 (pre-norm, 4 heads, BCE with pos_weight 5, AdamW lr 1e-3, weight decay 1e-2, batch 512, max 30 epochs, patience 5 on VAL-3 PR-AUC, gradient clip 1);
- selection ties: tolerance 0.05, simplest first (smaller W, then fewer parameters), then PR-AUC.

What is **not** done here: the CAL measurement, the N0/N1 adoption decision (H-B1/H-B2), the v1 comparison and H-B6. They are notebook 31. TEST is notebook 32.

GPU strongly recommended (T4 is fine). Every model is saved when it finishes; the notebook stops itself after `MAX_RUN_MINUTES` and continues on the next *Run all*.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
import torch, torch.nn as nn
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; TR = OUT/'transformer'; TR.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]; MODES = ['N0', 'N1']
MOCK = os.environ.get('TS_MOCK') == '1'
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time()
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_EPOCHS, PATIENCE, BATCH, LR, WD, POSW, NEG_FRAC = (2 if MOCK else 30), 5, 512, 1e-3, 1e-2, 5.0, (0.05 if MOCK else 0.5)
GRID = [(W, d, L) for W in [16, 32, 64] for (d, L) in [(32, 1), (64, 1), (64, 2)]]
if MOCK: GRID = [(16, 32, 1), (16, 64, 1), (32, 32, 1)]
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first (it freezes the protocol)'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol was changed'
for s in SYMBOLS:
    for m in MODES:
        assert (OUT/'scores'/f'ae_ctx_{s}_{m}.npz').exists(), 'run notebook 28 to the end first'
        assert (OUT/'scores'/f'xgb_v2_{s}_{m}.npz').exists(), 'run notebook 29 to the end first (XGBoost scores are used as a comparator)'
say(f'protocol B2 intact, sha256 {ph[:16]}; torch {torch.__version__}; device {DEV}' + ('' if DEV.type == 'cuda' else '  <<< NO GPU: switch the runtime to GPU for the real run >>>') + ('  [MOCK RUN]' if MOCK else ''))

## 1 · Library (same file as notebook 28)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

## 2 · Data and row masks

In [ ]:
LOOK = 15; FEATS = B.FEATS; BUDGETS = B.BUDGETS; PB = B.PRIMARY; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
D = {}; SCO = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet').iloc[:31*1440]      # genuine Aug only (to 2024-08-31): TEST-B is never loaded
    for d in (a, b): d.loc[d.index[:15], 'volume_acceleration'] = 0.0
    sets = {}
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
        sets[name] = {int(r): (g.reset_index(drop=True), ev[ev.replica == r].reset_index(drop=True)) for r, g in f.groupby('replica')}
    D[s] = {'GEN_L1': a[FEATS].reset_index(drop=True), 'GEN_AO': b[FEATS].reset_index(drop=True), 'sets': sets}
    for mode in MODES:
        z = np.load(OUT/'scores'/f'ae_ctx_{s}_{mode}.npz'); SCO[(s, mode)] = {k: z[k] for k in z.files if k != 'scale'}
    for mode in MODES:
        for k, v in SCO[(s, mode)].items():
            n_ = len(D[s]['GEN_L1']) if k == 'GEN_L1' else len(D[s]['GEN_AO']) if k == 'GEN_AO' else len(D[s]['sets'][k.split('_r')[0]][int(k.split('_r')[1])][0])
            assert len(v) == n_ and v.shape[1] == 8, (s, mode, k)
    say(f'{s}: genuine Layer 1 {len(a)} bars, genuine Aug {len(b)} bars; replicas ' + str({k: len(v) for k, v in sets.items()}))
assert not any('TEST' in k for s in SYMBOLS for m in MODES for k in SCO[(s, m)])
N0 = {s: B.fit_n0(D[s]['GEN_L1'].iloc[15:90720]) for s in SYMBOLS}

def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)
FA_NAMES = build_feats(np.zeros((200, 6)))[1]
NAMES = {'A': FA_NAMES, 'B': FA_NAMES + ['ae__fmax', 'ae__fmean'] + ['ae__e_' + f for f in FEATS]}; NCOL = {'A': len(FA_NAMES), 'B': len(FA_NAMES) + 8}
assert NCOL == {'A': 53, 'B': 61}
XS = {}; RZ = {}
def series_F(s, key):
    if key in ('GEN_L1', 'GEN_AO'): return D[s][key]
    name, r = key.split('_r'); return D[s]['sets'][name][int(r)][0]
def XOF(s, mode, key, cache=True):
    k = (s, mode, key)
    if k in XS: return XS[k]
    z = B.normalise(series_F(s, key), mode, N0[s]); X = np.hstack([build_feats(z)[0], SCO[(s, mode)][key].astype(np.float32)]); X[:B.WARM] = np.nan
    if key.startswith('VAL') or key == 'GEN_L1': RZ[k] = np.abs(z).max(axis=1).astype(np.float32)
    if cache and not key.startswith(('CAL', 'GEN_AO')): XS[k] = X
    return X
# row masks of the TRAIN-3 and VAL-3 replicas (label: 0 clean, 1..8 family, -1 tail)
RS = {}
for s in SYMBOLS:
    for name in ['TRAIN', 'VAL']:
        for r, (f, ev) in D[s]['sets'][name].items():
            lab = f.label.to_numpy().astype(int); inseg = f.in_seg.to_numpy().astype(bool); n = len(f); idx = np.arange(n)
            c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
            clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= B.WARM)      # no non-clean bar in [t-15, t]
            pos = (lab > 0) & inseg & (idx >= B.WARM); assert not (clean & pos).any(); RS[(s, name, r)] = (lab, inseg, clean, pos)
say('rows: ' + str({(s, n_): int(sum(RS[(s, n_, r)][3].sum() for r in D[s]['sets'][n_])) for s in SYMBOLS for n_ in ['TRAIN', 'VAL']}) + ' positive bars; clean negatives (replica 0) ' + str({(s, n_): int(RS[(s, n_, 0)][2].sum()) for s in SYMBOLS for n_ in ['TRAIN', 'VAL']}))

## 3 · Evaluation helpers (T0 / T1 / T1c, union, chance)

In [ ]:
XSC = {(s, m): dict(np.load(OUT/'scores'/f'xgb_v2_{s}_{m}.npz')) for s in SYMBOLS for m in MODES}
L1V = B.valid_mask(129600, 90720, 129600); PB = B.PRIMARY; BUDGETS = B.BUDGETS
def event_hits(alarm, ev):
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def macro_rate(dfh, fams): return float(np.mean([dfh[(dfh.sym == s) & dfh.fam.isin(fams)].groupby('fam').hit.mean().mean() for s in SYMBOLS]))
def thr_for(g, rule, bud):
    if rule == 'T0': return np.full(len(g), B.t0_threshold(g, L1V, bud))
    q = B.t1_q(g, L1V, bud); t_ = B.rolling_thr(g, q); return np.where(np.isnan(t_), np.inf, t_)
def hits_rule(get_score, rule, bud):
    """T0 static; T1c rolling threshold from the GENUINE series applied to the injected series (amendment A1); T1 = rolling threshold of the injected series itself (pre-registered)."""
    recs = []; dev = {}
    for s in SYMBOLS:
        g = get_score(s, 'GEN_L1').astype('float64'); thr_g = thr_for(g, 'T0' if rule == 'T0' else 'T1c', bud); dev[s] = B.eps_per_day(g > thr_g, L1V)
        for r, (f, ev) in D[s]['sets']['VAL'].items():
            sc_ = get_score(s, f'VAL_r{r}').astype('float64')
            if rule == 'T1': par = B.t1_q(g, L1V, bud); thr = np.where(np.isnan(B.rolling_thr(sc_, par)), np.inf, B.rolling_thr(sc_, par))
            else: thr = thr_g
            recs.append(pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy(), 'hit': event_hits(sc_ > thr, ev)}))
    return pd.concat(recs, ignore_index=True), dev
def hits_union(fns, rule, bud):
    recs = []; dev = {}
    for s in SYMBOLS:
        gs = [fn(s, 'GEN_L1').astype('float64') for fn in fns]; thrs = [thr_for(g, 'T0' if rule == 'T0' else 'T1c', bud) for g in gs]
        dev[s] = B.eps_per_day(np.any([g > t_ for g, t_ in zip(gs, thrs)], axis=0), L1V)
        for r, (f, ev) in D[s]['sets']['VAL'].items():
            al = np.any([fn(s, f'VAL_r{r}').astype('float64') > t_ for fn, t_ in zip(fns, thrs)], axis=0); recs.append(pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy(), 'hit': event_hits(al, ev)}))
    return pd.concat(recs, ignore_index=True), dev
def chance_recall(alarm_dev, evs, fams, nsh=100, seed=0):
    rg = np.random.default_rng(seed); out = []
    for _ in range(nsh):
        full = np.zeros(129600, bool); full[90720:] = np.roll(alarm_dev, int(rg.integers(0, len(alarm_dev)))); h = event_hits(full, evs)
        d_ = pd.DataFrame({'fam': evs.family.to_numpy(), 'h': h}); d_ = d_[d_.fam.isin(fams)]; out.append(d_.groupby('fam').h.mean().mean())
    return float(np.mean(out))
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
def ae_fn(mode): return lambda s, key: nanw(SCO[(s, mode)][key][:, 0])
def xgb_fn(mode): return lambda s, key: nanw(XSC[(s, mode)][key])
OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]
def series_F(s, key):
    if key in ('GEN_L1', 'GEN_AO'): return D[s][key]
    name, r = key.split('_r'); return D[s]['sets'][name][int(r)][0]
ZC = {}
def ZOF(s, mode, key):
    k = (s, mode, key)
    if k not in ZC: ZC[k] = B.normalise(series_F(s, key), mode, N0[s]).astype(np.float32)
    return ZC[k]

## 4 · Model, training, scoring

In [ ]:
class Net(nn.Module):
    def __init__(s, W, d, L, heads=4, drop=0.1):
        super().__init__(); s.inp = nn.Linear(6, d); s.pos = nn.Parameter(torch.zeros(1, W, d)); nn.init.normal_(s.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, batch_first=True, norm_first=True, activation='gelu')
        s.enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False); s.norm = nn.LayerNorm(d); s.out = nn.Linear(d, 1)
    def forward(s, x): return s.out(s.norm(s.enc(s.inp(x) + s.pos)[:, -1])).squeeze(-1)
def cid(W, d, L): return f'W{W}_d{d}_L{L}'
def nparams(W, d, L): return sum(p.numel() for p in Net(W, d, L).parameters())
def rows_W(s, name, r, W, lofo=None):
    """positives (old families only for VAL; LOFO family removed) and clean negatives for a window of W bars ending at t."""
    f, ev = D[s]['sets'][name][r]; lab = f.label.to_numpy().astype(int); inseg = f.in_seg.to_numpy().astype(bool); n = len(f); idx = np.arange(n); lo = max(B.WARM, W - 1)
    c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))]); clean = ((c[idx + 1] - c[np.maximum(idx - W + 1, 0)]) == 0) & inseg & (idx >= lo)
    pos = (lab > 0) & inseg & (idx >= lo)
    if name == 'VAL': pos &= (lab <= 6)
    if lofo is not None: pos &= (lab != lofo)
    return idx, lab, pos, clean
def build_pool(mode, W, lofo=None):
    """concatenated normalised series (TRAIN-3 / VAL-3 replicas, both symbols) on the device + global row indices and labels."""
    parts = []; off = 0; rowsT = []; rowsV = []; log = []
    for s in SYMBOLS:
        for name in ['TRAIN', 'VAL']:
            for r in D[s]['sets'][name]:
                z = ZOF(s, mode, f'{name}_r{r}'); idx, lab, pos, clean = rows_W(s, name, r, W, lofo); m = pos | (clean if r == 0 else np.zeros(len(lab), bool)); ii = np.flatnonzero(m)
                (rowsT if name == 'TRAIN' else rowsV).append((off + ii, pos[ii].astype(np.float32), r == 0 and False)); parts.append(z); log.append({'sym': s, 'set': name, 'replica': r, 'min_bar': int(ii.min()), 'max_bar': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(pos[ii].sum()), 'pos_labels': sorted(set(lab[ii][pos[ii]].tolist()))}); off += len(z)
    Z = torch.tensor(np.concatenate(parts), dtype=torch.float32, device=DEV)
    cat = lambda rr, k: np.concatenate([x[k] for x in rr])
    return Z, cat(rowsT, 0), cat(rowsT, 1), cat(rowsV, 0), cat(rowsV, 1), log
def gather(Z, gi, W):
    ar = torch.arange(-W + 1, 1, device=DEV); return Z[gi[:, None] + ar[None, :]]
@torch.no_grad()
def score_series(net, z, W, bs=8192):
    net.eval(); Zt = torch.tensor(z, dtype=torch.float32, device=DEV); n = len(z); out = np.full(n, np.nan, dtype=np.float32); lo = max(B.WARM, W - 1)
    for a in range(lo, n, bs):
        gi = torch.arange(a, min(n, a + bs), device=DEV); out[a:a + len(gi)] = net(gather(Zt, gi, W)).float().cpu().numpy()
    return out
def mpath(mode, W, d, L, seed, lofo=None): return TR/(f'{mode}_{cid(W, d, L)}_s{seed}' + ('' if lofo is None else f'_no{lofo}') + '.pt')
def train_one(mode, W, d, L, seed, lofo=None):
    p = mpath(mode, W, d, L, seed, lofo); ip = Path(str(p)[:-3] + '.info.json')
    if p.exists() and ip.exists(): return False
    torch.manual_seed(seed); rg = np.random.default_rng(seed); Z, gT, yT, gV, yV, log = build_pool(mode, W, lofo)
    net = Net(W, d, L).to(DEV); opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD); lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POSW, device=DEV))
    posT = np.flatnonzero(yT > 0); negT = np.flatnonzero(yT == 0)
    if MOCK: posT = rg.choice(posT, min(len(posT), 1500), replace=False)
    best = (-1.0, None, 0); bad = 0; t = time.time(); hist = []
    gVt = torch.tensor(gV, device=DEV)
    for ep in range(MAX_EPOCHS):
        net.train(); sel_ = np.concatenate([posT, rg.choice(negT, int(len(negT) * NEG_FRAC), replace=False)]); rg.shuffle(sel_); tl = 0.0
        for a in range(0, len(sel_), BATCH):
            b = sel_[a:a + BATCH]; x = gather(Z, torch.tensor(gT[b], device=DEV), W); y = torch.tensor(yT[b], device=DEV)
            opt.zero_grad(); loss = lossf(net(x), y); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); tl += float(loss) * len(b)
        net.eval(); sv = []
        with torch.no_grad():
            for a in range(0, len(gV), 8192): sv.append(net(gather(Z, gVt[a:a + 8192], W)).float().cpu().numpy())
        ap = float(average_precision_score(yV, np.concatenate(sv))); hist.append((ep, tl / len(sel_), ap))
        if ap > best[0]: best = (ap, {k: v.detach().clone() for k, v in net.state_dict().items()}, ep); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    torch.save(best[1], p); info = {'best_epoch': int(best[2]), 'best_val_prauc': best[0], 'epochs_run': len(hist), 'n_fit': int(len(yT)), 'n_fit_pos': int(yT.sum()), 'n_es': int(len(yV)), 'n_es_pos': int(yV.sum()), 'seconds': round(time.time() - t, 1), 'rows': log, 'hist': hist}
    ip.write_text(json.dumps(info)); say(f'  trained {p.stem}: best epoch {info["best_epoch"]} (of {len(hist)}), val PR-AUC {best[0]:.3f}, fit {len(yT):,} rows ({int(yT.sum()):,} positive), {info["seconds"]}s'); return True
class TModel:
    def __init__(self, mode, cfg, seeds, lofo=None): self.mode, self.cfg, self.seeds, self.lofo = mode, cfg, list(seeds), lofo; self.cache = {}; self.nets = None
    def load(self):
        if self.nets is None:
            W, d, L = self.cfg; self.nets = []
            for sd in self.seeds:
                n_ = Net(W, d, L).to(DEV); n_.load_state_dict(torch.load(mpath(self.mode, W, d, L, sd, self.lofo), map_location=DEV)); n_.eval(); self.nets.append(n_)
        return self.nets
    def score(self, s, key):
        k = (s, key)
        if k not in self.cache:
            W = self.cfg[0]; z = ZOF(s, self.mode, key); m = np.mean([score_series(n_, z, W) for n_ in self.load()], axis=0).astype(np.float32); m[:max(B.WARM, W - 1)] = np.nan; self.cache[k] = m
            if key.startswith(('CAL', 'GEN_AO')): ZC.pop((s, self.mode, key), None)
        return self.cache[k]
def pr_auc_val(model):
    v = []
    for s in SYMBOLS:
        ys, ss = [], []
        for r in D[s]['sets']['VAL']:
            lab, inseg, clean, pos = RS[(s, 'VAL', r)]; p = pos & (lab <= 6); m = p | (clean if r == 0 else np.zeros(len(lab), bool)); sc_ = model.score(s, f'VAL_r{r}'); m &= np.isfinite(sc_); ys.append(p[m]); ss.append(sc_[m])
        v.append(average_precision_score(np.concatenate(ys), np.concatenate(ss)))
    return float(np.mean(v))

## 5 · Unit tests

In [ ]:
res = {}; s0 = SYMBOLS[0]
# T01 window gather equals plain slicing
zt = ZOF(s0, 'N0', 'VAL_r0'); Zt = torch.tensor(zt, device=DEV); gi = torch.tensor([200, 5000, 70000], device=DEV)
res['T01 window gather equals plain slicing (W = 16, 32, 64)'] = all(bool(np.allclose(gather(Zt, gi, W).cpu().numpy()[k], zt[int(gi[k]) - W + 1:int(gi[k]) + 1])) for W in [16, 32, 64] for k in range(3))
# T02 model scores are causal: changing bars after t leaves the scores <= t unchanged
torch.manual_seed(0); net = Net(32, 32, 1).to(DEV); z1 = zt[:6000].copy(); z2 = z1.copy(); z2[3500:] *= 3
a1 = score_series(net, z1, 32); a2 = score_series(net, z2, 32); res['T02 Transformer scores are causal'] = bool(np.allclose(a1[:3500], a2[:3500], equal_nan=True, atol=1e-5))
# T03 clean negatives for a window W contain no non-clean bar in [t-W+1, t]; positives are never from the new families in fit/ES rows
ok3 = True
for W in [16, 64]:
    idx, lab, pos, clean = rows_W(s0, 'TRAIN', 0, W); ii = np.flatnonzero(clean)[::997]; ok3 &= all((lab[max(0, t - W + 1):t + 1] == 0).all() for t in ii) and not (pos & clean).any()
    idx, lab, pos, clean = rows_W(s0, 'VAL', 0, W); ok3 &= not any(l > 6 for l in set(lab[pos].tolist()))
res['T03 clean negatives have a clean window of length W; VAL rows exclude the new families'] = bool(ok3)
# T04 LOFO removes the held-out family from the positives
idx, lab, pos, clean = rows_W(s0, 'TRAIN', 0, 16, lofo=3); res['T04 LOFO removes the held-out family from the positive rows'] = bool(3 not in set(lab[pos].tolist()) and 3 in set(rows_W(s0, 'TRAIN', 0, 16)[1][rows_W(s0, 'TRAIN', 0, 16)[2]].tolist()))
# T05 recall rule against a plain loop
rng = np.random.default_rng(0); al = rng.random(5000) < 0.01; evt = pd.DataFrame({'t0': rng.integers(100, 4000, 80), 'dur': rng.integers(1, 40, 80)})
res['T05 event detection equals a plain loop'] = bool((event_hits(al, evt) == np.array([bool(al[r.t0:r.t0 + r.dur].any()) for r in evt.itertuples()])).all())
# T06 chance baseline limits
evt2 = pd.DataFrame({'t0': rng.integers(90800, 129000, 60), 'dur': rng.integers(1, 30, 60), 'family': rng.integers(1, 7, 60)})
res['T06 chance baseline: no alarms -> 0, alarms everywhere -> 1'] = bool(chance_recall(np.zeros(38880, bool), evt2, OLD, 5) == 0.0 and chance_recall(np.ones(38880, bool), evt2, OLD, 5) == 1.0)
# T07 same seed reproduces the scores on a tiny fit (Spearman >= 0.98)
def tiny(seed):
    torch.manual_seed(seed); n_ = Net(16, 32, 1).to(DEV); o_ = torch.optim.AdamW(n_.parameters(), lr=1e-3); x = Zt[None, :1000].repeat(1, 1, 1)[0]
    for _ in range(20): o_.zero_grad(); l_ = n_(gather(Zt, torch.arange(100, 612, device=DEV), 16)).pow(2).mean(); l_.backward(); o_.step()
    return score_series(n_, zt[:3000], 16)
t1_, t2_ = tiny(5), tiny(5); mk = np.isfinite(t1_); res['T07 same seed reproduces scores (Spearman >= 0.98)'] = bool(spearmanr(t1_[mk], t2_[mk])[0] >= 0.98)
# T08 the comparator score files align with the series
res['T08 AE and XGBoost comparator scores have one value per bar'] = all(len(XSC[(s, m)]['GEN_L1']) == 129600 and len(XSC[(s, m)]['VAL_r0']) == len(D[s]['sets']['VAL'][0][0]) for s in SYMBOLS for m in MODES)
for k_, v in res.items(): say(('PASS ' if v else 'FAIL ') + k_)
assert all(res.values())

## 6 · Selection on VAL-3 (9 candidates × N0/N1)

In [ ]:
say('--- selection on VAL-3 (seed 11); metric = recall macro over the 6 old families, mean BTC/ETH, T1c (amendment A2), 2 episodes/day ---')
SEL = {}; SEL_DONE = True
for mode in MODES:
    for cfg in GRID:
        jf = TR/f'sel_{mode}_{cid(*cfg)}.json'
        if jf.exists(): SEL[(mode, cfg)] = json.loads(jf.read_text()); continue
        if out_of_time(): SEL_DONE = False; continue
        train_one(mode, *cfg, 11); m = TModel(mode, cfg, [11])
        h1c, dev1c = hits_rule(m.score, 'T1c', PB); h1, _ = hits_rule(m.score, 'T1', PB); h0, _ = hits_rule(m.score, 'T0', PB)
        r = {'mode': mode, 'config': cid(*cfg), 'W': cfg[0], 'd': cfg[1], 'L': cfg[2], 'params': nparams(*cfg), 'metric_T1c': macro_rate(h1c, OLD), 'metric_T1': macro_rate(h1, OLD), 'metric_T0': macro_rate(h0, OLD), 'recall_new2_T1c': macro_rate(h1c, NEW), 'pr_auc': pr_auc_val(m), 'dev_eps_T1c': float(np.mean(list(dev1c.values())))}
        jf.write_text(json.dumps(r)); SEL[(mode, cfg)] = r; say(f'{mode} {cid(*cfg):12s} VAL-3 recall(old6) T1c {r["metric_T1c"]:.3f}  T1 {r["metric_T1"]:.3f}  T0 {r["metric_T0"]:.3f}  PR-AUC {r["pr_auc"]:.3f}  DEV eps/day {r["dev_eps_T1c"]:.2f}')
        m.nets = None; torch.cuda.empty_cache() if DEV.type == 'cuda' else None
if not SEL_DONE: say('>>> time budget reached: Run all again to continue (finished models are kept). The rest is skipped until all %d candidates are done. <<<' % (2*len(GRID)))

## 7 · Selection rule (amendment A2)

In [ ]:
SELECTED = {}; BEST16 = {}; TAB = None
def pick(tab, col):
    best = tab[col].max(); tied = tab[tab[col] >= best - 0.05].copy(); tied['k'] = list(zip(tied.W, tied.params, -tied.pr_auc)); return tied.sort_values('k').iloc[0]
if SEL_DONE:
    TAB = pd.DataFrame(list(SEL.values())); TAB.to_csv(OUT/'transformer_selection_table.csv', index=False); say('--- selection table ---'); say(TAB.round(3).to_string(index=False))
    for mode in MODES:
        t_ = TAB[TAB['mode'] == mode]; row = pick(t_, 'metric_T1c'); SELECTED[mode] = (int(row.W), int(row.d), int(row.L)); r16 = pick(t_[t_.W == 16], 'metric_T1c'); BEST16[mode] = (16, int(r16.d), int(r16.L))
        say(f'{mode}: selected {row.config} (T1c {row.metric_T1c:.3f}); best W=16 candidate {cid(*BEST16[mode])}; selection by T1 would give {pick(t_, "metric_T1").config}, by T0 {pick(t_, "metric_T0").config}')
    SAME = all(pick(TAB[TAB['mode'] == m_], 'metric_T1c').config == pick(TAB[TAB['mode'] == m_], 'metric_T1').config for m_ in MODES); say('selection under T1c ' + ('equals' if SAME else 'DIFFERS from') + ' the T1 selection (reported, no switch)')
    TAB2 = pd.read_csv(OUT/'transformer_selection_table.csv'); assert all(pick(TAB2[TAB2['mode'] == m_], 'metric_T1c').config == cid(*SELECTED[m_]) for m_ in MODES) and len(TAB2) == 2 * len(GRID)
    say('PASS T09 the selection is reproduced from the saved table alone (%d candidates)' % (2*len(GRID)))

## 8 · H-B3 (window) with leave-one-family-out

In [ ]:
FINAL = {}; HB3 = {}; LOFO_DONE = SEL_DONE
if SEL_DONE:
    say('--- H-B3: leave-one-family-out for the selected candidate and the best W=16 candidate (seed 11) ---')
    for mode in MODES:
        cfgs = [SELECTED[mode]] + ([BEST16[mode]] if SELECTED[mode][0] != 16 else [])
        for cfg in cfgs:
            for fam in OLD:
                if out_of_time(): LOFO_DONE = False; break
                train_one(mode, *cfg, 11, lofo=fam)
    if not LOFO_DONE: say('>>> time budget reached: Run all again to continue. <<<')
LOFO = {}
def lofo_macro(mode, cfg, rule):
    out = {}
    for fam in OLD:
        ml = TModel(mode, cfg, [11], lofo=fam); dl, _ = hits_rule(ml.score, rule, PB); out[fam] = float(dl[dl.fam == fam].hit.mean())
    return out
if LOFO_DONE and SEL_DONE:
    for mode in MODES:
        sel, b16 = SELECTED[mode], BEST16[mode]; LOFO[(mode, sel)] = lofo_macro(mode, sel, 'T1c')
        if sel[0] == 16: FINAL[mode] = sel; HB3[mode] = {'selected_W': 16, 'decision': 'W = 16 selected directly; H-B3 not needed'}
        else:
            LOFO[(mode, b16)] = lofo_macro(mode, b16, 'T1c'); ms, mb = np.mean(list(LOFO[(mode, sel)].values())), np.mean(list(LOFO[(mode, b16)].values())); ok = bool(ms - mb >= 0.05)
            FINAL[mode] = sel if ok else b16; HB3[mode] = {'selected_W': sel[0], 'lofo_macro_selected': float(ms), 'lofo_macro_best16': float(mb), 'difference': float(ms - mb), 'H-B3 (>= 0.05)': ok, 'final': cid(*FINAL[mode])}
        say(f'{mode}: H-B3 -> {HB3[mode]}; final candidate {cid(*FINAL[mode])}')

## 9 · Final seeds

In [ ]:
FIN_DONE = LOFO_DONE and SEL_DONE
if FIN_DONE:
    say('--- final seeds ---')
    for mode in MODES:
        for sd in SEEDS[1:]:
            if out_of_time(): FIN_DONE = False; break
            train_one(mode, *FINAL[mode], sd)
    if not FIN_DONE: say('>>> time budget reached: Run all again to continue. <<<')
    else:
        ok = True
        for p in TR.glob('*.info.json'):
            inf = json.loads(p.read_text()); nm = p.name
            for r in inf['rows']:
                if r['set'] == 'TRAIN': ok &= not any(l > 6 for l in r['pos_labels'])
                if r['set'] == 'VAL': ok &= not any(l > 6 for l in r['pos_labels'])
                if '_no' in nm:
                    held = int(nm.split('_no')[1].split('.')[0]); ok &= held not in r['pos_labels']
        say(('PASS' if ok else 'FAIL') + ' T10 no row of a new family (or of the held-out family) in any fit or early-stopping set'); assert ok

## 10 · VAL-3 evaluation, comparators, union, chance, LOFO, per family

In [ ]:
if FIN_DONE:
    FM = {m: TModel(m, FINAL[m], SEEDS) for m in MODES}; ROWS = []; HC = {}
    for mode in MODES:
        comps = {'Transformer v2': FM[mode].score, 'XGBoost v2': xgb_fn(mode), 'AE-ctx fmax': ae_fn(mode)}
        for mname, fn in comps.items():
            for rule in ['T1c', 'T1', 'T0']:
                for bud in (BUDGETS if rule == 'T1c' else [PB]):
                    dfh, dev = hits_rule(fn, rule, bud); HC[(mode, mname, rule, bud)] = dfh; po = dfh[dfh.fam.isin(OLD)].hit.to_numpy(); pn = dfh[dfh.fam.isin(NEW)].hit.to_numpy()
                    ROWS.append({'norm': mode, 'model': mname, 'thr': rule, 'budget/day': bud, 'DEV genuine eps/day': float(np.mean(list(dev.values()))), 'recall 6 old': macro_rate(dfh, OLD), 'old CI': tuple(np.round(B.bootstrap_ci(po), 3)), 'recall 2 new': macro_rate(dfh, NEW), 'new CI': tuple(np.round(B.bootstrap_ci(pn), 3))})
        du, dev_u = hits_union(list(comps.values()), 'T1c', PB); HC[(mode, 'Union of three', 'T1c', PB)] = du
        ROWS.append({'norm': mode, 'model': 'Union of three (each at 2/day)', 'thr': 'T1c', 'budget/day': PB, 'DEV genuine eps/day': float(np.mean(list(dev_u.values()))), 'recall 6 old': macro_rate(du, OLD), 'old CI': tuple(np.round(B.bootstrap_ci(du[du.fam.isin(OLD)].hit.to_numpy()), 3)), 'recall 2 new': macro_rate(du, NEW), 'new CI': tuple(np.round(B.bootstrap_ci(du[du.fam.isin(NEW)].hit.to_numpy()), 3))})
    VT = pd.DataFrame(ROWS); VT.to_csv(OUT/'transformer_val_table.csv', index=False); say('--- VAL-3 recall (T1c = deployable emulation with clean history, T1 = pre-registered, T0 = static) ---'); say(VT.round(3).to_string(index=False))
    say('--- chance baseline at 2 episodes/day under T1c (same alarms at random times) ---'); CH = {}
    for mode in MODES:
        for mname, fn in {'Transformer v2': FM[mode].score, 'XGBoost v2': xgb_fn(mode), 'AE-ctx fmax': ae_fn(mode)}.items():
            ro, rn = [], []
            for s in SYMBOLS:
                g = fn(s, 'GEN_L1').astype('float64'); a_dev = (g > thr_for(g, 'T1c', PB))[90720:]; evs = pd.concat([ev for r, (f, ev) in D[s]['sets']['VAL'].items()], ignore_index=True); ro.append(chance_recall(a_dev, evs, OLD)); rn.append(chance_recall(a_dev, evs, NEW))
            CH[(mode, mname)] = (float(np.mean(ro)), float(np.mean(rn))); say(f'{mode} {mname}: chance recall 6 old {np.mean(ro):.3f}, 2 new {np.mean(rn):.3f}')
    PF = []
    for mode in MODES:
        for mname in ['Transformer v2', 'XGBoost v2', 'AE-ctx fmax', 'Union of three']:
            d_ = HC[(mode, mname, 'T1c', PB)]
            for fam, v_ in d_.groupby('fam').hit.mean().items(): PF.append({'norm': mode, 'model': mname, 'family': B.FAMS[int(fam)], 'recall': v_})
    PFT = pd.DataFrame(PF); PFT.to_csv(OUT/'transformer_val_by_family.csv', index=False); say('--- per family, T1c, 2 episodes/day ---'); say(PFT.pivot_table(index='family', columns=['norm', 'model'], values='recall').round(2).to_string())
    LR = []
    for mode in MODES:
        lf = LOFO[(mode, FINAL[mode])]
        for fam in OLD: LR.append({'norm': mode, 'family': B.FAMS[fam], 'Transformer without this family': lf[fam], 'Transformer trained on it': float(HC[(mode, 'Transformer v2', 'T1c', PB)].pipe(lambda d_: d_[d_.fam == fam].hit.mean())), 'XGBoost v2 trained on it': float(HC[(mode, 'XGBoost v2', 'T1c', PB)].pipe(lambda d_: d_[d_.fam == fam].hit.mean())), 'AE-ctx fmax': float(HC[(mode, 'AE-ctx fmax', 'T1c', PB)].pipe(lambda d_: d_[d_.fam == fam].hit.mean()))})
    LT = pd.DataFrame(LR); LT.to_csv(OUT/'transformer_lofo_table.csv', index=False); say('--- leave-one-family-out, T1c, 2 episodes/day ---'); say(LT.round(3).to_string(index=False)); say('LOFO macro:'); say(LT.groupby('norm')[['Transformer without this family', 'Transformer trained on it', 'XGBoost v2 trained on it', 'AE-ctx fmax']].mean().round(3).to_string())
    HB5 = {}
    for mode in MODES:
        rz_new = None; b = {m_: macro_rate(HC[(mode, m_, 'T1c', PB)], NEW) for m_ in ['Transformer v2', 'XGBoost v2', 'AE-ctx fmax']}; HB5[mode] = {'recall_new2_T1c': b, 'best_trained_minus_chance': float(max(b['Transformer v2'], b['XGBoost v2']) - CH[(mode, 'XGBoost v2')][1])}
    say('H-B5 indicative (unseen families, T1c): ' + json.dumps(HB5))

## 11 · Save scores and decisions

In [ ]:
if FIN_DONE:
    for mode in MODES:
        for s in SYMBOLS:
            keys = ['GEN_L1', 'GEN_AO'] + [f'{n_}_r{r}' for n_ in ['TRAIN', 'VAL', 'CAL'] for r in D[s]['sets'][n_]]
            np.savez_compressed(OUT/'scores'/f'transformer_v2_{s}_{mode}.npz', **{k: FM[mode].score(s, k) for k in keys})
    say('scores saved: models_b2_v1/scores/transformer_v2_{symbol}_{N0|N1}.npz')
    DEC = {'selected': {m: cid(*SELECTED[m]) for m in MODES}, 'final': {m: cid(*FINAL[m]) for m in MODES}, 'H-B3': HB3, 'H-B5 indicative on VAL-3': HB5, 'chance': {f'{k[0]}|{k[1]}': v for k, v in CH.items()},
           'amendment_A2': 'selection by T1c; negatives subsampled 50% per epoch', 'note': 'CAL, N0/N1 adoption (H-B1/H-B2), v1 comparison and H-B6 are notebook 31; TEST untouched.'}
    (OUT/'transformer_v2_decisions.json').write_text(json.dumps(DEC, indent=1, default=str)); say('saved transformer_v2_decisions.json, transformer_selection_table.csv, transformer_val_table.csv, transformer_lofo_table.csv, transformer_val_by_family.csv')

## 12 · Share block

In [ ]:
block = '\n'.join(LINES); (OUT/'b2_step4_share_block.txt').write_text(block)
tail = block[-14000:]; tail = tail[tail.find('\n') + 1:] if len(block) > 14000 else block
print('\n===== SHARE BLOCK (best: attach the file models_b2_v1/b2_step4_share_block.txt; otherwise copy everything below) =====\n' + tail)